In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [10]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

current_directory = os.getcwd()
if current_directory.endswith("RecSys_Course_AT_PoliMi"):
    print("Already in the correct directory.")
    pass
else:
    os.chdir("RecSys_Course_AT_PoliMi")

!pwd
!python run_compile_all_cython.py

Already in the correct directory.
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi
zsh:1: command not found: python


In [11]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender

In [12]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [13]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [ ]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali
URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [15]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri 
        recommender_instance = P3alphaRecommender(URM_combined)
        recommender_instance.fit(alpha = optuna_trial.suggest_float("alpha", 0.09, 0.17, log=True),
                                 topK =  optuna_trial.suggest_int("topK", 75, 115),
                                 normalize_similarity= True,
                                 )
        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        scores.append(result["RECALL"].values[0])
        
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [16]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

[I 2025-12-02 17:00:36,535] A new study created in memory with name: no-name-0863cde2-0a84-4d7e-88de-b7c7fb4f6a0d


P3alphaRecommender: Similarity column 6969 (100.0%), 2932.28 column/sec. Elapsed time 2.38 sec
EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


[W 2025-12-02 17:00:41,099] Trial 0 failed with parameters: {'alpha': 0.1590504836569961, 'topK': 114} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_43922/1392254058.py", line 36, in objective_function_funksvd
    result, _ = evaluator_test.evaluateRecommender(recommender_instance)
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/Evaluation/Evaluator.py", line 276, in evaluateRecommender
    results_dict = self._run_evaluation_on_selected_users(recommender_object, self.users_to_evaluate)
  File "/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi/Evaluation/E

KeyboardInterrupt: 

In [ ]:
optuna_study.best_trial.params

In [ ]:
save_results.results_df

# Saving trials and hyperparameters
df_trials = optuna_study.trials_dataframe()

# Filtra colonne che ci interessano
df_filtered = df_trials[['number', 'value', 'params_alpha', 'params_topK']].copy()
# Rinomina le colonne
df_filtered.columns = ['trial_id', 'recall', 'alpha', 'topK']
df_best_5 = df_filtered.sort_values(by='recall', ascending=False).head(5)

# 4. Visualizza e/o Salva

print("------------ 🔝 5 BEST TRIALS WITH HYPERPARAMETERS (BY RECALL) ------")
print(df_best_5)  # Stampa solo i 5 migliori trial
print("------------------------")


In [ ]:
#/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("Already in the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results")

df_filtered.to_csv("p3alpha_par_opt_results_v2.csv", index=False)

!pwd
!python run_compile_all_cython.py

In [ ]:
optuna_study.best_value
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
print("-----------best hyperparameters-----------\n")
print(best_hyperparams)